# 42 — Risky interactions (Group C)

The combinations that were real bugs, each as a probit and a logit study. One of them (item covariate + free b) estimates κ only through b's prior, so its intervals are wide and it is centred on what the I items can tell (the regression of the item shifts on Q), not exactly on the true κ; compare and judge.

## C1. Shared b across nodes (pooled difficulty)

## Setup — install the package, load the harness simulator

In [ ]:
import sys
!{sys.executable} -m pip install -e "C:\Users\manonfi\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\PACKAGE\Irtree"

In [ ]:
import sys, warnings, pathlib, importlib.util
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec   as ts
import irtree.node_models as nm
import irtree.priors      as pr
import irtree.constraints as con
import irtree.ingest      as ing
import irtree.fit         as ft

# The data-generating code (simulate_irtree) lives in the test harness, NOT in
# the package: simulation is deliberately kept separate from fitting.  Load it
# the same way notebooks 40 & 50 do.
ROOT = pathlib.Path.cwd().parents[1]
_hspec = importlib.util.spec_from_file_location("_harness", ROOT / "tests" / "_harness.py")
h = importlib.util.module_from_spec(_hspec); _hspec.loader.exec_module(h)

# probit -> logit conversion, per parameter role (TESTING.md 9.3), for your
# own hand comparisons between the two fits.
D_PL = 1.702
def to_logit(value, role):
    if role == 'slope':        return D_PL * np.asarray(value)
    if role in ('location','correlation','difficulty'): return np.asarray(value)
    raise ValueError(role)

## Study size (you choose)

In [ ]:
# ===================== study size — set these yourself =====================
# One simulated dataset per scenario (a rehearsal / recovery study, not a tiered
# test). Evidence-mode warmup is the package default, so any fit with an MH block
# (logit nodes, 3PL guessing, an estimated correlation) needs N_ITER >= ~1600;
# pure-Gibbs probit Rasch has no such floor.
P, I     = 1500, 30
N_ITER   = 4000
N_CHAINS = 2
SEED     = 20250413
print(f'P={P}  I={I}  N_ITER={N_ITER}  N_CHAINS={N_CHAINS}  SEED={SEED}')

## Two small helpers for this notebook
`responses_frame` makes the simulated matrix look like a real response file for `ingest`; `compare` prints posterior vs truth. Everything else is called explicitly so you can rehearse the API.

In [ ]:
def responses_frame(Y):
    """Turn a simulated [P, I] response matrix into the kind of keyed table a
    real study would hand to ingest.load_data: one row per person, item ids as
    column headers, an explicit person_id. (load_data also reads a CSV path.)"""
    Y = np.asarray(Y); Pn, In = Y.shape
    df = pd.DataFrame(Y, columns=[f'item{i+1}' for i in range(In)])
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(Pn)])
    return df

def person_frame(X, names):
    df = pd.DataFrame(np.asarray(X), columns=list(names))
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(np.asarray(X).shape[0])])
    return df

def item_frame(Q, names):
    df = pd.DataFrame(np.asarray(Q), columns=list(names))
    df.insert(0, 'item_id', [f'item{i+1}' for i in range(np.asarray(Q).shape[0])])
    return df

def compare(true_vec, prefix, fit, label=''):
    """Posterior mean + 95% interval for every parameter named `prefix*`,
    next to the known truth. Rehearses pulling estimates out of fit.results.
    Returns the DataFrame so you can compare however you like."""
    names = fit.results[0].param_names
    cols  = [j for j, n in enumerate(names) if n.startswith(prefix)]
    draws = np.concatenate([np.asarray(r.chain, float) for r in fit.results])[:, cols]
    tbl = pd.DataFrame({
        'param'    : [names[j] for j in cols],
        'true'     : np.asarray(true_vec, float).ravel()[:len(cols)],
        'post_mean': draws.mean(0),
        'ci_2.5'   : np.percentile(draws, 2.5, axis=0),
        'ci_97.5'  : np.percentile(draws, 97.5, axis=0),
    })
    tbl['abs_err'] = (tbl['post_mean'] - tbl['true']).abs()
    if label: print(label)
    print(tbl.to_string(index=False,
          formatters={c: '{:+.3f}'.format for c in ['true','post_mean','ci_2.5','ci_97.5','abs_err']}))
    return tbl

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+20)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b_shared_true = rng.normal(0.3, 0.8, I)                      # SAME b at both nodes
truth = [dict(theta=theta1, b=b_shared_true), dict(theta=theta2, b=b_shared_true)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+20), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+20), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
cons = con.Constraints(b_shared=[[1, 2]])                    # nodes 1 & 2 share one b array
fit = ft.irtree_fit(data, spec_dis, node_models, constraints=cons, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+20)
pref = [p for p in ['b_1&2_','b_1_'] if any(n.startswith(p) for n in fit.results[0].param_names)][0]
print('shared b stored under prefix:', pref)
compare(b_shared_true, pref, fit, f'shared difficulties (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, categories_start=1)
node_models = [nm.LogitRaschNode(), nm.LogitRaschNode()]
cons = con.Constraints(b_shared=[[1, 2]])
fit = ft.irtree_fit(data, spec_dis, node_models, constraints=cons, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+20)
pref = [p for p in ['b_1&2_','b_1_'] if any(n.startswith(p) for n in fit.results[0].param_names)][0]
compare(b_shared_true, pref, fit, f'shared difficulties (logit)')

## C2. 2PL node + person covariate

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+21)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)
beta_true = np.array([0.5, -0.4])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true; theta2 = rng.normal(0,1,P)
b1, b2 = rng.normal(0.6,0.7,I), rng.normal(0,1,I)
a2_true = rng.uniform(0.8,1.5,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2, a=a2_true)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+21), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+21), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
xdf = person_frame(Xmat, ['escs','female'])

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, person_covariates=xdf, categories_start=1)
node_models = [nm.ProbitRaschNode(person_covariates=['escs','female']), nm.ProbitTwoPLNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+21)
compare(beta_true, 'beta_', fit, 'person covariate beta (probit)')
compare(a2_true,  'a_2_', fit, 'node-2 discrimination a (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, person_covariates=xdf, categories_start=1)
node_models = [nm.LogitRaschNode(person_covariates=['escs','female']), nm.LogitTwoPLNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+21)
compare(beta_true, 'beta_', fit, 'person covariate beta (logit)')
compare(a2_true,  'a_2_', fit, 'node-2 discrimination a (logit, MH)')

## C3. Item covariate + free b — κ identified through b's prior (weakly)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+22)
Qmat = rng.normal(size=(I,2)); kappa_true = np.array([0.4, -0.3])
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1, b2 = rng.normal(0,0.7,I), rng.normal(0,1,I)
off1 = (Qmat @ kappa_true)[None, :].repeat(P, 0)             # item covariate enters node-1 difficulty
truth = [dict(theta=theta1, b=b1, offset=off1), dict(theta=theta2, b=b2)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+22), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+22), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
qdf = item_frame(Qmat, ['q0','q1'])
# NB: b_i and Q_i·kappa are both item shifts.  kappa is identified through b's
# prior: it is the regression of the item shifts (Q kappa - b) on Q, with the b_i
# as residuals.  With I items that regression is what the data can tell, so the
# posterior centres on it rather than on kappa_true (TESTING.md §5):
kappa_target = np.linalg.lstsq(Qmat, Qmat @ kappa_true - b1, rcond=None)[0]
print('true kappa', kappa_true, ' regression of the true item shifts on Q', np.round(kappa_target, 3))

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, item_covariates=qdf, categories_start=1)
node_models = [nm.ProbitRaschNode(item_covariates=['q0','q1']), nm.ProbitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+22)
compare(kappa_true, 'kappa_', fit, 'item covariate kappa (probit; compare with kappa_target too)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, item_covariates=qdf, categories_start=1)
node_models = [nm.LogitRaschNode(item_covariates=['q0','q1']), nm.LogitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+22)
compare(kappa_true, 'kappa_', fit, 'item covariate kappa (logit; compare with kappa_target too)')

## C4. Polytomous node + person covariate — GRM (probit) / GPCM (logit)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_tri = ts.disengagement_tree(binary=False)
rng = np.random.default_rng(SEED+23)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)
beta_true = np.array([0.5, -0.4])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true; theta2 = rng.normal(0,1,P)
b1_true = rng.normal(0.6,0.7,I); thr_true = np.sort(rng.normal(0,1,(I,2)), axis=1)
truth = [dict(theta=theta1, b=b1_true), dict(theta=theta2, b=thr_true)]
Y_probit = h.simulate_irtree(spec_tri, truth, P, I, np.random.default_rng(SEED+23),
                             node_types=['binary','grm'],  node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_tri, truth, P, I, np.random.default_rng(SEED+23),
                             node_types=['binary','gpcm'], node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
xdf = person_frame(Xmat, ['escs','female'])

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_tri, person_covariates=xdf, categories_start=1)
node_models = [nm.ProbitRaschNode(person_covariates=['escs','female']), nm.GRMNode(n_outcomes=3)]
fit = ft.irtree_fit(data, spec_tri, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+23)
compare(beta_true, 'beta_', fit, 'person covariate beta under GRM node (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_tri, person_covariates=xdf, categories_start=1)
node_models = [nm.LogitRaschNode(person_covariates=['escs','female']), nm.GPCMNode(n_outcomes=3)]
fit = ft.irtree_fit(data, spec_tri, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+23)
compare(beta_true, 'beta_', fit, 'person covariate beta under GPCM node (logit)')

## C5. Person covariate + 10% missing X (imputed each iteration)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+24)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)
beta_true = np.array([0.5, -0.4])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true; theta2 = rng.normal(0,1,P)
b1, b2 = rng.normal(0.6,0.7,I), rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+24), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+24), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
Xm = Xmat.copy(); miss = rng.choice(P, int(P*0.1), replace=False); Xm[miss, 0] = 99.0   # sentinel
xdf_missing = person_frame(Xm, ['escs','female'])

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, person_covariates=xdf_missing,
                     categories_start=1, covariate_missing_codes=(99,))
node_models = [nm.ProbitRaschNode(person_covariates=['escs','female']), nm.ProbitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+24)
compare(beta_true, 'beta_', fit, 'beta with 10% missing X imputed (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, person_covariates=xdf_missing,
                     categories_start=1, covariate_missing_codes=(99,))
node_models = [nm.LogitRaschNode(person_covariates=['escs','female']), nm.LogitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+24)
compare(beta_true, 'beta_', fit, 'beta with 10% missing X imputed (logit)')